In [ ]:
!pip3 install aiortc opencv-python 

In [ ]:
# Install ultralytics
!pip install ultralytics --no-deps

# Install torch and torchvision (cpu)
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install other core dependencies
!pip install numpy matplotlib polars pyyaml pillow psutil requests scipy ultralytics-thop

# Install headless OpenCV instead of the default
!pip install opencv-python-headless

In [ ]:
from ultralytics import YOLO

# Configure the tracking parameters and run the tracker
model = YOLO("yolo11n.pt")
results = model.track(source="https://youtu.be/LNwODJXcvt4", conf=0.3, iou=0.5, show=True)

In [ ]:
# Export a YOLO11n PyTorch model to ONNX format
!yolo export model=yolo11n.pt format=onnx # creates 'yolo11n.onnx'

# Run inference with the exported model
!yolo predict model=yolo11n.onnx source='https://ultralytics.com/images/bus.jpg'

In [ ]:
!pip install "fastapi[standard]"

In [1]:
from ultralytics import YOLO

model = 'yolo11n.onnx'

In [2]:
from fastapi import FastAPI, UploadFile
from fastapi.middleware.cors import CORSMiddleware
import numpy as np
import cv2

app = FastAPI()

# Allow phone access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/upload")
async def upload(frame: UploadFile):
    # Read JPEG bytes
    data = await frame.read()
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)

    # Run YOLO
    results = model.track(img, verbose=False)

    # Print detections (debug)
    print(results[0].boxes.xyxy)

    return {"ok": True}

